# Results notebook

Read-only: loads the frozen pickles in `results/` and formats the paper's tables. No experiment is run, no model is trained, no random number is drawn here. If you want to regenerate the underlying numbers, use the scripts in `scripts/`, not this notebook.

Run from the repository root (`jupyter notebook notebooks/results.ipynb`), or set `EDGEGIRTH_RESULTS_DIR` to point at a different results directory (e.g. one produced by `scripts/reproduce_all.sh`) before launching.

In [ ]:
import os
import pickle

import pandas as pd

RESULTS_DIR = os.environ.get("EDGEGIRTH_RESULTS_DIR", os.path.join(os.pardir, "results"))


def load(name):
    path = os.path.join(RESULTS_DIR, name)
    with open(path, "rb") as f:
        return pickle.load(f)


print(f"Reading from: {os.path.abspath(RESULTS_DIR)}")

## ZINC-12k main table

In [ ]:
zinc = load("zinc_results.pkl")
print(f"mode={zinc['mode']}, seeds={zinc['seeds']}, epoch={zinc['epoch']}")

rows = []
for method, r in zinc["results"].items():
    rows.append(
        {
            "method": method,
            "hidden_dim": r["hidden_dim"],
            "n_params": r["n_params"],
            "mean_test_mae": r["mean_test_mae"],
            "std_test_mae": r["std_test_mae"],
            "per_seed_mae": [round(s["test_mae"], 4) for s in r["seed_results"]],
        }
    )
pd.DataFrame(rows).sort_values("mean_test_mae").reset_index(drop=True)

## ZINC ablation

In [ ]:
ablation = load("ablation_results.pkl")
order = ["full", "g_only", "lambda_only", "constant", "noise"]
rows = []
for variant in order:
    r = ablation["results"][variant]
    rows.append(
        {
            "variant": variant,
            "mean_test_mae": r["mean_test_mae"],
            "std_test_mae": r["std_test_mae"],
            "per_seed_mae": [round(s["test_mae"], 4) for s in r["seed_results"]],
        }
    )
pd.DataFrame(rows)

## Descriptor-vs-descriptor study

`constant`/`full` are reused from `ablation_results.pkl` (same architecture, same regime, not retrained separately).

In [ ]:
descriptor = load("descriptor_study_results.pkl")
cold = load("descriptor_study_cold_cache_preprocessing.pkl")

combined = dict(descriptor["results"])
for variant in ("constant", "full"):
    combined[variant] = ablation["results"][variant]

order = ["constant", "tri", "cyc4", "cyc6", "cyc8", "full"]
rows = []
for variant in order:
    r = combined[variant]
    preproc = cold["results"].get(variant, {}).get("preprocessing_seconds")
    rows.append(
        {
            "variant": variant,
            "hidden_dim": r["hidden_dim"],
            "n_params": r["n_params"],
            "mean_test_mae": r["mean_test_mae"],
            "std_test_mae": r["std_test_mae"],
            "per_seed_mae": [round(s["test_mae"], 4) for s in r["seed_results"]],
            "cold_cache_preproc_seconds": preproc,
        }
    )
pd.DataFrame(rows)

## BREC

In [ ]:
brec = load("brec_results.pkl")
cats = list(next(iter(brec["results"].values()))["by_category"].keys())
rows = []
for method, r in brec["results"].items():
    row = {"method": method}
    for cat in cats:
        row[cat] = r["by_category"][cat]["accuracy"]
    row["overall"] = r["overall_accuracy"]
    row["reliability"] = 1.0 - r["n_reliable_fail"] / r["n_pairs"]
    rows.append(row)
pd.DataFrame(rows).sort_values("overall", ascending=False).reset_index(drop=True)

## Contingency tables (BREC annex): global and Regular-category stratification

In [ ]:
global_contingency = load("global_contingency.pkl")
print(f"n_total={global_contingency['n_total']}, n_egr={global_contingency['n_egr']}")
pd.DataFrame(global_contingency["contingency_by_method"]).T

In [ ]:
strat = load("regular_stratified_analysis.pkl")
print(
    f"reference_method={strat['reference_method']}, n_egr={strat['n_egr']}, "
    f"n_sr={strat['n_sr']}, egr_sr_coincide={strat['egr_sr_coincide']}"
)
pd.DataFrame(
    {"plain_regular_not_sr": strat["plain_contingency"], "strongly_regular": strat["strong_contingency"]}
).T

## CSL (annex) -- two methods are marked unreliable, see below

In [ ]:
csl = load("csl_results.pkl")
unreliable = csl.get("unreliable_methods", {})
rows = []
for method, r in csl["results"].items():
    rows.append(
        {
            "method": method,
            "mean_acc": r["mean_acc"],
            "std_acc": r["std_acc"],
            "unreliable": method in unreliable,
        }
    )
display(pd.DataFrame(rows).sort_values("mean_acc", ascending=False).reset_index(drop=True))
for method, reason in unreliable.items():
    print(f"\n{method}: {reason}")